In [31]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

In [32]:
CLASS_NAMES=['baklawa', 'chourba', 'couscous', 'egg', 'french fries', 'pizza', 'sfenje', 'spaghetti', 'sushi', 'tajin_zitoun', 'tiramisu']

In [33]:
from PIL import Image
import onnxruntime as ort

session = ort.InferenceSession('./final_export/food_classifier.onnx')

def predict(image_path):
    img = Image.open(image_path).convert('RGB')
    img = img.resize((224, 224))
    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - np.array(MEAN, dtype=np.float32)) / np.array(STD, dtype=np.float32)
    arr = arr.transpose(2, 0, 1)[np.newaxis].astype(np.float32)  # ← explicit cast
    logits = session.run(None, {'input': arr})[0]
    probs = np.exp(logits) / np.exp(logits).sum()
    top_idx = probs.argmax()
    return {
        'class': CLASS_NAMES[top_idx],
        'confidence': f"{probs[0][top_idx]*100:.1f}%",
        'top3': [(CLASS_NAMES[i], f"{probs[0][i]*100:.1f}%")
                  for i in probs[0].argsort()[::-1][:3]]
    }

In [35]:
import numpy as np

predict("./photo_2026-05-23_01-26-50.jpg")

{'class': 'sfenje',
 'confidence': '85.3%',
 'top3': [('sfenje', '85.3%'), ('baklawa', '1.8%'), ('tajin_zitoun', '1.7%')]}

In [43]:
!pip install "numpy<2.0"

   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/15.8 MB 19.1 MB/s eta 0:00:01
   ----------------------- ---------------- 9.4/15.8 MB 29.3 MB/s eta 0:00:01
   ------------------------------------- -- 14.9/15.8 MB 28.4 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 27.6 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2


    extract-msg (<=0.29.*)
                 ~~~~~~~^
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
object-detection 0.0.3 requires tensorflow, which is not installed.
tensorflow-intel 2.17.0 requires keras>=3.2.0, which is not installed.
mediapipe 0.10.14 requires protobuf<5,>=4.25.3, but you have protobuf 3.20.3 which is incompatible.
fiftyone 0.25.2 requires dacite<1.8.0,>=1.6.0, but you have dacite 1.9.2 which is incompatible.
yfinance 0.2.66 requires beautifulsoup4>=4.11.1, but you have beautifulsoup4 4.8.2 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 12.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
D:\anaconda3\envs\my_env\lib\site-pac

In [2]:
import numpy as np
import json
from PIL import Image
import tensorflow as tf

# Load model & class names
interpreter = tf.lite.Interpreter(model_path='./final_export/tflite_model/food_classifier_float32.tflite')
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:",  input_details[0]['shape'])   # should be [1, 224, 224, 3] or [1, 3, 224, 224]
print("Input dtype:",  input_details[0]['dtype'])   # should be float32
print("Output shape:", output_details[0]['shape'])  # should be [1, 11]

with open('./final_export/class_names.json') as f:
    CLASS_NAMES = json.load(f)

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def predict_tflite(image_path):
    # Preprocess
    img = Image.open(image_path).convert('RGB').resize((224, 224))
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - MEAN) / STD                        # HWC still

    # Check input layout from model
    if input_details[0]['shape'][1] == 3:
        arr = arr.transpose(2, 0, 1)                # → CHW if model expects that
    arr = arr[np.newaxis]                           # → [1, C, H, W] or [1, H, W, C]

    # Run
    interpreter.set_tensor(input_details[0]['index'], arr)
    interpreter.invoke()
    logits = interpreter.get_tensor(output_details[0]['index'])[0]

    # Softmax
    exps = np.exp(logits - logits.max())            # subtract max for numerical stability
    probs = exps / exps.sum()

    top3_idx = probs.argsort()[::-1][:3]

    return {
        'class':      CLASS_NAMES[probs.argmax()],
        'confidence': f"{probs.max()*100:.1f}%",
        'top3': [(CLASS_NAMES[i], f"{probs[i]*100:.1f}%") for i in top3_idx]
    }

# Test
result = predict_tflite('./photo_2026-05-23_01-26-50.jpg')
print(result)

Input shape: [  1 224 224   3]
Input dtype: <class 'numpy.float32'>
Output shape: [ 1 11]
{'class': 'sfenje', 'confidence': '85.3%', 'top3': [('sfenje', '85.3%'), ('baklawa', '1.8%'), ('tajin_zitoun', '1.7%')]}
